In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import math
import torch
import wikipedia
from newspaper import Article, ArticleException
from GoogleNews import GoogleNews
import IPython
from pyvis.network import Network

/~/HDD1/miniconda3/envs/rebel/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")

In [3]:
def extract_relations_from_model_output(text):
    relations = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    text_replaced = text.replace("<s>", "").replace("<pad>", "").replace("</s>", "")
    for token in text_replaced.split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                relations.append({
                    'head': subject.strip(),
                    'type': relation.strip(),
                    'tail': object_.strip()
                })
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                relations.append({
                    'head': subject.strip(),
                    'type': relation.strip(),
                    'tail': object_.strip()
                })
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        relations.append({
            'head': subject.strip(),
            'type': relation.strip(),
            'tail': object_.strip()
        })
    return relations

In [4]:
class KB():
    def __init__(self):
        # self.relations = []
        # self.entities = {}
        self.entities = {} # { entity_title: {...} }
        self.relations = [] # [ head: entity_title, type: ..., tail: entity_title,
          # meta: { article_url: { spans: [...] } } ]
        self.sources = {} # { article_url: {...} }


    def are_relations_equal(self, r1, r2):
        return all(r1[attr] == r2[attr] for attr in ["head", "type", "tail"])

    def exists_relation(self, r1):
        return any(self.are_relations_equal(r1, r2) for r2 in self.relations)
            
    def merge_relations(self, r1):
        r2 = [r for r in self.relations
              if self.are_relations_equal(r1, r)][0]
        spans_to_add = [span for span in r1["meta"]["spans"]
                        if span not in r2["meta"]["spans"]]
        r2["meta"]["spans"] += spans_to_add

    # def merge_relations(self, r2):
    #     r1 = [r for r in self.relations
    #           if self.are_relations_equal(r2, r)][0]

    #     # if different article
    #     article_url = list(r2["meta"].keys())[0]
    #     if article_url not in r1["meta"]:
    #         r1["meta"][article_url] = r2["meta"][article_url]

    #     # if existing article
    #     else:
    #         spans_to_add = [span for span in r2["meta"][article_url]["spans"]
    #                         if span not in r1["meta"][article_url]["spans"]]
    #         r1["meta"][article_url]["spans"] += spans_to_add

    def get_wikipedia_data(self, candidate_entity):
        try:
            page = wikipedia.page(candidate_entity, auto_suggest=False)
            entity_data = {
                "title": page.title,
                "url": page.url,
                "summary": page.summary
            }
            return entity_data
        except:
            return None

    def add_entity(self, e):
        self.entities[e["title"]] = {k:v for k,v in e.items() if k != "title"}

    def add_relation(self, r):
        # check on wikipedia
        candidate_entities = [r["head"], r["tail"]]
        entities = [self.get_wikipedia_data(ent) for ent in candidate_entities]

        # if one entity does not exist, stop
        if any(ent is None for ent in entities):
            return

        # manage new entities
        for e in entities:
            self.add_entity(e)

        # rename relation entities with their wikipedia titles
        r["head"] = entities[0]["title"]
        r["tail"] = entities[1]["title"]

        # manage new relation
        if not self.exists_relation(r):
            self.relations.append(r)
        else:
            self.merge_relations(r)

    def print(self):
        print("Entities:")
        for e in self.entities.items():
            print(f"  {e}")
        print("Relations:")
        for r in self.relations:
            print(f"  {r}")

    def merge_with_kb(self, kb2):
        for r in kb2.relations:
            article_url = list(r["meta"].keys())[0]
            source_data = kb2.sources[article_url]
            self.add_relation(r, source_data["article_title"],
                              source_data["article_publish_date"])

    # def add_relation(self, r, article_title, article_publish_date):
    #     # check on wikipedia
    #     candidate_entities = [r["head"], r["tail"]]
    #     entities = [self.get_wikipedia_data(ent) for ent in candidate_entities]

    #     # if one entity does not exist, stop
    #     if any(ent is None for ent in entities):
    #         return

    #     # manage new entities
    #     for e in entities:
    #         self.add_entity(e)

    #     # rename relation entities with their wikipedia titles
    #     r["head"] = entities[0]["title"]
    #     r["tail"] = entities[1]["title"]

    #     # add source if not in kb
    #     article_url = list(r["meta"].keys())[0]
    #     if article_url not in self.sources:
    #         self.sources[article_url] = {
    #             "article_title": article_title,
    #             "article_publish_date": article_publish_date
    #         }

    #     # manage new relation
    #     if not self.exists_relation(r):
    #         self.relations.append(r)
    #     else:
    #         self.merge_relations(r)

    def print(self):
        print("Entities:")
        for e in self.entities.items():
            print(f"  {e}")
        print("Relations:")
        for r in self.relations:
            print(f"  {r}")
        print("Sources:")
        for s in self.sources.items():
            print(f"  {s}")

In [5]:
def from_small_text_to_kb(text, verbose=False):
    kb = KB()

    # Tokenizer text
    model_inputs = tokenizer(text, max_length=512, padding=True, truncation=True,
                            return_tensors='pt')
    if verbose:
        print(f"Num tokens: {len(model_inputs['input_ids'][0])}")

    # Generate
    gen_kwargs = {
        "max_length": 216,
        "length_penalty": 0,
        "num_beams": 3,
        "num_return_sequences": 3
    }
    generated_tokens = model.generate(
        **model_inputs,
        **gen_kwargs,
    )
    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

    # create kb
    for sentence_pred in decoded_preds:
        relations = extract_relations_from_model_output(sentence_pred)
        for r in relations:
            kb.add_relation(r)

    return kb

In [6]:
def from_text_to_kb(text, span_length=128, verbose=False):
# def from_text_to_kb(text, article_url, span_length=128, article_title=None,
#                     article_publish_date=None, verbose=False):    
    # tokenize whole text
    inputs = tokenizer([text], return_tensors="pt")

    # compute span boundaries
    num_tokens = len(inputs["input_ids"][0])
    if verbose:
        print(f"Input has {num_tokens} tokens")
    num_spans = math.ceil(num_tokens / span_length)
    if verbose:
        print(f"Input has {num_spans} spans")
    overlap = math.ceil((num_spans * span_length - num_tokens) / 
                        max(num_spans - 1, 1))
    spans_boundaries = []
    start = 0
    for i in range(num_spans):
        spans_boundaries.append([start + span_length * i,
                                 start + span_length * (i + 1)])
        start -= overlap
    if verbose:
        print(f"Span boundaries are {spans_boundaries}")

    # transform input with spans
    tensor_ids = [inputs["input_ids"][0][boundary[0]:boundary[1]]
                  for boundary in spans_boundaries]
    tensor_masks = [inputs["attention_mask"][0][boundary[0]:boundary[1]]
                    for boundary in spans_boundaries]
    inputs = {
        "input_ids": torch.stack(tensor_ids),
        "attention_mask": torch.stack(tensor_masks)
    }

    # generate relations
    num_return_sequences = 3
    gen_kwargs = {
        "max_length": 256,
        "length_penalty": 0,
        "num_beams": 3,
        "num_return_sequences": num_return_sequences
    }
    generated_tokens = model.generate(
        **inputs,
        **gen_kwargs,
    )

    # decode relations
    decoded_preds = tokenizer.batch_decode(generated_tokens,
                                           skip_special_tokens=False)

    # create kb
    kb = KB()
    i = 0
    for sentence_pred in decoded_preds:
        current_span_index = i // num_return_sequences
        relations = extract_relations_from_model_output(sentence_pred)
        for relation in relations:
            relation["meta"] = {
                "spans": [spans_boundaries[current_span_index]]
            }
            kb.add_relation(relation)
        i += 1
    # for sentence_pred in decoded_preds:
    #     current_span_index = i // num_return_sequences
    #     relations = extract_relations_from_model_output(sentence_pred)
    #     for relation in relations:
    #         relation["meta"] = {
    #             article_url: {
    #                 "spans": [spans_boundaries[current_span_index]]
    #             }
    #         }
    #         kb.add_relation(relation, article_title, article_publish_date)
    #     i += 1


    return kb

In [7]:
def get_article(url):
    article = Article(url)
    article.download()
    article.parse()
    return article

def from_url_to_kb(url):
    article = get_article(url)
    config = {
        "article_title": article.title,
        "article_publish_date": article.publish_date
    }
    kb = from_text_to_kb(article.text, article.url, **config)
    return kb

In [8]:
def get_news_links(query, lang="en", region="US", pages=1, max_links=100000):
    googlenews = GoogleNews(lang=lang, region=region)
    googlenews.search(query)
    all_urls = []
    for page in range(pages):
        googlenews.get_page(page)
        all_urls += googlenews.get_links()
    return list(set(all_urls))[:max_links]

def from_urls_to_kb(urls, verbose=False):
    kb = KB()
    if verbose:
        print(f"{len(urls)} links to visit")
    for url in urls:
        if verbose:
            print(f"Visiting {url}...")
        try:
            kb_url = from_url_to_kb(url)
            kb.merge_with_kb(kb_url)
        except ArticleException:
            if verbose:
                print(f"  Couldn't download article at url {url}")
    return kb

In [9]:
def save_network_html(kb, filename="network.html"):
    # create network
    net = Network(directed=True, width="700px", height="700px", bgcolor="#eeeeee")

    # nodes
    color_entity = "#00FF00"
    for e in kb.entities:
        net.add_node(e, shape="circle", color=color_entity)

    # edges
    for r in kb.relations:
        net.add_edge(r["head"], r["tail"],
                    title=r["type"], label=r["type"])
        
    # save network
    net.repulsion(
        node_distance=200,
        central_gravity=0.2,
        spring_length=200,
        spring_strength=0.05,
        damping=0.09
    )
    net.set_edge_smooth('dynamic')
    # with open(filename, 'w', encoding='utf-8') as f:
    #     f.write(net.get_html())
    # net.show(filename)
    net.save_graph(filename)

In [10]:
# news_links = get_news_links("Google", pages=1, max_links=3)
# kb = from_urls_to_kb(news_links, verbose=True)
# kb.print()

In [11]:
# url = "https://finance.yahoo.com/news/microstrategy-bitcoin-millions-142143795.html"
# kb = from_url_to_kb(url)
# kb.print()


# Entities:
#   ('MicroStrategy', {'url': 'https://en.wikipedia.org/wiki/MicroStrategy',
#     'summary': "MicroStrategy Incorporated is an American company that ..."})
#   ('Michael J. Saylor', {'url': 'https://en.wikipedia.org/wiki/Michael_J._Saylor',
#     'summary': 'Michael J. Saylor (born February 4, 1965) is an American ..."})
#   ...
# Relations:
#   {'head': 'MicroStrategy', 'type': 'founded by', 'tail': 'Michael J. Saylor',
#    'meta': {'https://finance.yahoo.com/news/microstrategy-bitcoin-millions-142143795.html': 
#      {'spans': [[0, 128]]}}}
#   {'head': 'Michael J. Saylor', 'type': 'employer', 'tail': 'MicroStrategy',
#    'meta': {'https://finance.yahoo.com/news/microstrategy-bitcoin-millions-142143795.html':
#      {'spans': [[0, 128]]}}}
#   ...
# Sources:
#   ('https://finance.yahoo.com/news/microstrategy-bitcoin-millions-142143795.html',
#     {'article_title': "Microstrategy chief: 'Bitcoin is going to go into the millions'",
#      'article_publish_date': None})

In [12]:
# text = """
# Napoleon Bonaparte (born Napoleone di Buonaparte; 15 August 1769 – 5 May 1821), and later known by his regnal name Napoleon I, was a French military and political leader who rose to prominence during the French Revolution and led several successful campaigns during the Revolutionary Wars. He was the de facto leader of the French Republic as First Consul from 1799 to 1804. As Napoleon I, he was Emperor of the French from 1804 until 1814 and again in 1815. Napoleon's political and cultural legacy has endured, and he has been one of the most celebrated and controversial leaders in world history. Napoleon was born on the island of Corsica not long after its annexation by the Kingdom of France.[5] He supported the French Revolution in 1789 while serving in the French army, and tried to spread its ideals to his native Corsica. He rose rapidly in the Army after he saved the governing French Directory by firing on royalist insurgents. In 1796, he began a military campaign against the Austrians and their Italian allies, scoring decisive victories and becoming a national hero. Two years later, he led a military expedition to Egypt that served as a springboard to political power. He engineered a coup in November 1799 and became First Consul of the Republic. Differences with the British meant that the French faced the War of the Third Coalition by 1805. Napoleon shattered this coalition with victories in the Ulm Campaign, and at the Battle of Austerlitz, which led to the dissolving of the Holy Roman Empire. In 1806, the Fourth Coalition took up arms against him because Prussia became worried about growing French influence on the continent. Napoleon knocked out Prussia at the battles of Jena and Auerstedt, marched the Grande Armée into Eastern Europe, annihilating the Russians in June 1807 at Friedland, and forcing the defeated nations of the Fourth Coalition to accept the Treaties of Tilsit. Two years later, the Austrians challenged the French again during the War of the Fifth Coalition, but Napoleon solidified his grip over Europe after triumphing at the Battle of Wagram. Hoping to extend the Continental System, his embargo against Britain, Napoleon invaded the Iberian Peninsula and declared his brother Joseph King of Spain in 1808. The Spanish and the Portuguese revolted in the Peninsular War, culminating in defeat for Napoleon's marshals. Napoleon launched an invasion of Russia in the summer of 1812. The resulting campaign witnessed the catastrophic retreat of Napoleon's Grande Armée. In 1813, Prussia and Austria joined Russian forces in a Sixth Coalition against France. A chaotic military campaign resulted in a large coalition army defeating Napoleon at the Battle of Leipzig in October 1813. The coalition invaded France and captured Paris, forcing Napoleon to abdicate in April 1814. He was exiled to the island of Elba, between Corsica and Italy. In France, the Bourbons were restored to power. However, Napoleon escaped Elba in February 1815 and took control of France.[6][7] The Allies responded by forming a Seventh Coalition, which defeated Napoleon at the Battle of Waterloo in June 1815. The British exiled him to the remote island of Saint Helena in the Atlantic, where he died in 1821 at the age of 51. Napoleon had an extensive impact on the modern world, bringing liberal reforms to the many countries he conquered, especially the Low Countries, Switzerland, and parts of modern Italy and Germany. He implemented liberal policies in France and Western Europe.
# """

# kb = from_text_to_kb(text, verbose=True)
# kb.print()
# Input has 726 tokens
# Input has 6 spans
# Span boundaries are [[0, 128], [119, 247], [238, 366], [357, 485], [476, 604], [595, 723]]
# Relations:
#   {'head': 'Napoleon Bonaparte', 'type': 'date of birth',
#    'tail': '15 August 1769', 'meta': {'spans': [[0, 128]]}}
#   ...
#   {'head': 'Napoleon', 'type': 'place of birth',
#    'tail': 'Corsica', 'meta': {'spans': [[119, 247]]}}
#   ...
#   {'head': 'Fourth Coalition', 'type': 'start time',
#    'tail': '1806', 'meta': {'spans': [[238, 366]]}}
#   ...

In [13]:
# news_links = get_news_links("Google", pages=5, max_links=20)
# kb = from_urls_to_kb(news_links, verbose=True)

# save html
# filename = "network_3_google.html"
# save_network_html(kb, filename=filename)




In [14]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
import pandas_access as mdb

db_filename = './NTSBData/avall.mdb'

# Listing the tables.
print('Showing list of tables in the NSTB MS Access database:')
for tbl in mdb.list_tables(db_filename):
  print(tbl)

# Read the narratives table only.
df = mdb.read_table(db_filename, "narratives")
df.keys()
df = df[df['narr_accp'].notna()]
# df


Showing list of tables in the NSTB MS Access database:
Country
ct_iaids
ct_seqevt
dt_events
dt_Flight_Crew
eADMSPUB_DataDictionary
engines
events
Events_Sequence
Flight_Crew
flight_time
injury
NTSB_Admin
Occurrences
seq_of_events
states
aircraft
dt_aircraft
Findings
narratives


In [15]:
# column 'narr_accp'
concatenated_text = df['narr_accp'][:20].str.cat(sep=' ')
# print(concatenated_text)

In [16]:
kb = from_text_to_kb(concatenated_text, verbose=True)
kb.print()
filename = "network_aircraft.html"
save_network_html(kb, filename=filename)

Token indices sequence length is longer than the specified maximum sequence length for this model (13175 > 1024). Running this sequence through the model will result in indexing errors


Input has 13175 tokens
Input has 103 spans
Span boundaries are [[0, 128], [127, 255], [254, 382], [381, 509], [508, 636], [635, 763], [762, 890], [889, 1017], [1016, 1144], [1143, 1271], [1270, 1398], [1397, 1525], [1524, 1652], [1651, 1779], [1778, 1906], [1905, 2033], [2032, 2160], [2159, 2287], [2286, 2414], [2413, 2541], [2540, 2668], [2667, 2795], [2794, 2922], [2921, 3049], [3048, 3176], [3175, 3303], [3302, 3430], [3429, 3557], [3556, 3684], [3683, 3811], [3810, 3938], [3937, 4065], [4064, 4192], [4191, 4319], [4318, 4446], [4445, 4573], [4572, 4700], [4699, 4827], [4826, 4954], [4953, 5081], [5080, 5208], [5207, 5335], [5334, 5462], [5461, 5589], [5588, 5716], [5715, 5843], [5842, 5970], [5969, 6097], [6096, 6224], [6223, 6351], [6350, 6478], [6477, 6605], [6604, 6732], [6731, 6859], [6858, 6986], [6985, 7113], [7112, 7240], [7239, 7367], [7366, 7494], [7493, 7621], [7620, 7748], [7747, 7875], [7874, 8002], [8001, 8129], [8128, 8256], [8255, 8383], [8382, 8510], [8509, 8637], [

/~/HDD1/miniconda3/envs/rebel/lib/python3.9/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /~/HDD1/miniconda3/envs/rebel/lib/python3.9/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Entities:
  ('Napa County Airport', {'url': 'https://en.wikipedia.org/wiki/Napa_County_Airport', 'summary': "Napa County Airport (IATA: APC, ICAO: KAPC, FAA LID: APC) (Napa Valley Airport) is a public airport five miles (8 km) south of Napa, in Napa County, California, United States. It has three runways.\nDuring 2008 the airport's 1960-era control tower received extensive radio, plumbing and electrical upgrades and renovations funded by the federal government.\nAirport officials said the airport had about 122,000 flights take off or land at the facility annually in 2008."})
  ('Airport', {'url': 'https://en.wikipedia.org/wiki/Airport', 'summary': 'An airport is an aerodrome with extended facilities, mostly for commercial air transport.  Airports usually consist of a landing area, which comprises an aerially accessible open space including at least one operationally active surface such as a runway for a plane to take off and to land or a helipad, and often includes adjacent utility bui